In [1]:
# imports

# to get my token from .env
from dotenv import load_dotenv

# get token
import os

# handle requests 
import requests 

In [2]:
# get token from .env

# load .env
load_dotenv()
# get token
TOKEN = os.getenv("GITHUB_TOKEN")

# ensure TOKEN exists
print(TOKEN is not None)

True


In [3]:
# declare headers with TOKEN

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

In [4]:
try:
    # pipeline vars
    # get users after id 0
    since = 0
    # number of pages to ingest
    pages = 2

    # pagination size
    users_per_page = 30
    repos_per_page = 10
    commits_per_page = 20
    issues_per_page = 20
    
    # store user data
    all_users_data = []

    # store repos data
    all_repos_data = []

    # store repo commits data
    all_commits_data = []

    # store repo issues data
    all_issues_data = []

    # for each page get x users
    for i in range(pages):

        # get users data
        url = f"https://api.github.com/users?since={since}&per_page={users_per_page}"
        r = requests.get(url, headers=headers)

        # check if there is an error
        r.raise_for_status()

        # store json data
        users = r.json()

        # break if users is empty list > stop pagination loop
        if not users:
            break

        # get the usernames to get more data
        logins = [user["login"] for user in users]

        # for each user get more detailed data
        for login in logins:
            
            # url for specific user data using username
            url = f"https://api.github.com/users/{login}"
            user_r = requests.get(url, headers=headers)

            user_r.raise_for_status()

            # returns a dict per user
            user_data = user_r.json()

            # append user dict to all data
            all_users_data.append(user_data)

            # repo section
            # get 10 repos per user
            url = f"https://api.github.com/users/{login}/repos?per_page={repos_per_page}"
            repo_r = requests.get(url, headers=headers)

            repo_r.raise_for_status()

            # returns a list
            repo_data = repo_r.json()

            # extend repos data list to include repo data
            all_repos_data.extend(repo_data)
            
            # for each repo get the commits/issues for that repo
            for repo in repo_data:

                # get repo name for url
                repo_name = repo["name"]

                url = f"https://api.github.com/repos/{login}/{repo_name}/commits?per_page={commits_per_page}"
                commit_r = requests.get(url, headers=headers)

                # if 409 status code then commit history not accessible
                # so skip
                if commit_r.status_code == 409:
                    continue
                # check for other errors 
                else:
                    commit_r.raise_for_status()

                # returns a list
                commit_data = commit_r.json()

                all_commits_data.extend(commit_data)

                # this section is to ingest issues data
                
                # if repo has no issues then skip
                if repo["has_issues"] == False:
                    continue
                
                url = f"https://api.github.com/repos/{login}/{repo_name}/issues?per_page={issues_per_page}"

                issue_r = requests.get(url, headers=headers)

                issue_r.raise_for_status()

                issue_data = issue_r.json()

                # add issue data to all issues data
                all_issues_data.extend(issue_data)
        
        # GitHub REST api with users endpoint uses 'since' to get the users where the id is after 'since'
        # therefore, get the last id of the current call to get the next page
        since = users[-1]["id"]

# handle HTTP status errors first, then catch other request failures
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    
except requests.exceptions.RequestException as e:
    print("Other request error:", e)

In [5]:
print(len(all_users_data))
print(len(all_repos_data))
print(len(all_commits_data))
print(len(all_issues_data))

60
530
7657
424
